# Pré-processamento — Detecção de Fraude em Cartão de Crédito

CC0121 — Inteligência Artificial — UNIFAP — 2026.1

Objetivo: preparar os dados para modelagem —
1. Separar features (X) e alvo (y)
2. Normalizar `Amount` e `Time`
3. Split treino/teste estratificado
4. Salvar os conjuntos prontos para a etapa de modelagem

**Importante:** o balanceamento com SMOTE fica para o próximo notebook (`03_modeling.ipynb`), e será aplicado **apenas no conjunto de treino**. Fazer isso aqui, antes do split, causaria vazamento de dados (data leakage).

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

## 2. Carregar os dados

In [ ]:
df = pd.read_csv("../data/creditcard.csv")
df.shape

## 3. Separar features (X) e alvo (y)

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

print(X.shape, y.shape)
print(y.value_counts(normalize=True))

## 4. Split treino/teste estratificado

Usamos `stratify=y` para garantir que a proporção de fraudes (0,17%) seja mantida tanto no treino quanto no teste. Fazemos o split **antes** de qualquer normalização ou balanceamento, para evitar que informação do teste "vaze" para o treino.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Treino:", X_train.shape, "| Fraudes:", y_train.sum(), f"({y_train.mean()*100:.3f}%)")
print("Teste: ", X_test.shape, "| Fraudes:", y_test.sum(), f"({y_test.mean()*100:.3f}%)")

## 5. Normalizar `Amount` e `Time`

As colunas `V1`...`V28` já vêm padronizadas pelo PCA original do dataset. Só `Amount` e `Time` estão em escalas muito diferentes (`Amount` em reais/euros, `Time` em segundos), então aplicamos `StandardScaler` só nelas.

**Regra de ouro:** o `scaler` é ajustado (`fit`) somente com dados de **treino**, e depois aplicado (`transform`) em treino e teste. Isso evita vazamento de informação do teste.

In [ ]:
cols_to_scale = ["Amount", "Time"]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

X_train_scaled[cols_to_scale].describe()

## 6. Salvar os conjuntos processados

Salvamos em `data/processed/` para o notebook de modelagem carregar diretamente, sem repetir esses passos.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

X_train_scaled.to_csv("../data/processed/X_train.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

# Salvar o scaler para reusar no app Streamlit depois
joblib.dump(scaler, "../models/scaler.pkl")

print("Arquivos salvos em data/processed/ e models/scaler.pkl")

## 7. Próximo passo

Seguir para `03_modeling.ipynb`:
- Carregar `X_train`, `y_train`, `X_test`, `y_test` de `data/processed/`
- Aplicar SMOTE **somente em `X_train`/`y_train`**
- Treinar Regressão Logística, Random Forest e XGBoost
- Avaliar com Stratified K-Fold + Precision/Recall/F1/AUC-ROC